<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Análisis de Salida: Simulaciones Terminales
### T3.3 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Terminal vs. estacionaria — ¿por qué importa?
2. Autocorrelación: por qué S/√m no funciona
3. Método de réplicas independientes
4. **Ejemplo 1:** Banco M/M/1 terminal — 10 réplicas, IC
5. Determinación de n* (procedimiento secuencial)
6. **Ejemplo 2:** Banco M/E₂/1 — cálculo de n* para δ=0.05
7. **Ejemplo 3:** Urgencias hospitalarias — 3 prioridades, múltiples métricas

## Terminal vs. estacionaria
<div class='defn'>
<strong>Terminal:</strong> horizonte finito con evento de cierre natural. Las condiciones iniciales son parte del modelo.<br>
<strong>Estacionaria:</strong> comportamiento de largo plazo, independiente de condiciones iniciales.
</div>

| | Terminal | Estacionaria |
|---|---|---|
| **Ejemplos** | Turno de banco, jornada de urgencias | Fábrica 24/7, call center |
| **Inicio** | Estado fijo (p.ej. vacío) | Arbitrario (warm-up) |
| **Método** | Réplicas independientes | Batch Means (T3.4) |

<div class='alerta'>
⚠️ Usar el método equivocado produce conclusiones inválidas. Un turno de banco no se puede analizar como estado estacionario.
</div>

## ¿Por qué S/√m no funciona?

Dentro de una corrida, las observaciones Y₁, Y₂, ... están **autocorrelacionadas**:

$$\text{Var}[\bar{Y}_m] = \frac{\sigma^2}{m}\left[1 + 2\sum_{k=1}^{m-1}\left(1-\frac{k}{m}\right)\rho(k)\right]$$

Con autocorrelación positiva (típica en colas): **Var[Ȳ] >> σ²/m**

<div class='alerta'>
Si usas S/√m dentro de una sola corrida, la cobertura real del IC al "95%" puede ser del <strong>50–70%</strong>.
</div>

**Solución:** réplicas independientes → cada réplica produce un solo Y_j → los Y_j sí son i.i.d.

## Ejemplo 1 — Banco M/M/1 terminal (4 horas)
<div class='defn'>
Ventanilla bancaria: turno de 9:00 a 13:00 (4 horas = 240 min).
λ=2 cl/min, μ=3 cl/min, ρ=0.667. Sistema inicia vacío.
Estimar W_q con IC 95%.
</div>

**Referencia analítica** (estacionario, para validación):
$$W_q^{M/M/1} = \frac{\rho}{\mu(1-\rho)} = \frac{0.667}{3 \times 0.333} = 0.667 \text{ min}$$

*Nota: la simulación terminal incluye el transitorio (inicio vacío), así que Ȳ será algo menor que 0.667.*

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as st

def simular_turno_mm1(lam, mu, T, seed):
    """Simula un turno terminal M/M/1. Retorna Wq promedio y # clientes."""
    np.random.seed(seed)
    esperas = []
    
    def cliente(env, srv):
        t_arr = env.now
        with srv.request() as req:
            yield req
            esperas.append(env.now - t_arr)
            yield env.timeout(np.random.exponential(1/mu))
    
    def llegadas(env, srv):
        while True:
            yield env.timeout(np.random.exponential(1/lam))
            env.process(cliente(env, srv))
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)
    env.process(llegadas(env, srv))
    env.run(until=T)
    return np.mean(esperas), len(esperas)

lam, mu = 2, 3
T_turno = 240  # 4 horas en minutos
Wq_ref = (lam/mu) / (mu * (1-lam/mu))

# ─── 10 réplicas ───
n = 10
Y = np.zeros(n)
m_clientes = np.zeros(n, dtype=int)
for j in range(n):
    Y[j], m_clientes[j] = simular_turno_mm1(lam, mu, T_turno, seed=100+j)

Ybar = Y.mean()
S_n = Y.std(ddof=1)
t_crit = st.t.ppf(0.975, n-1)
h_n = t_crit * S_n / np.sqrt(n)
ci = (Ybar - h_n, Ybar + h_n)
r_n = h_n / abs(Ybar)

print('═══ EJEMPLO 1: BANCO M/M/1 TERMINAL (n=10 réplicas) ═══')
print(f'\n{"j":>3} {"Yj (min)":>10} {"# clientes":>12}')
print('-'*28)
for j in range(n):
    print(f'{j+1:>3} {Y[j]:>10.3f} {m_clientes[j]:>12}')

print(f'\nȲ = {Ybar:.3f} min')
print(f'S = {S_n:.3f} min')
print(f't(0.025, {n-1}) = {t_crit:.3f}')
print(f'Semiancho h = {h_n:.3f} min')
print(f'IC 95%: [{ci[0]:.3f}, {ci[1]:.3f}] min')
print(f'Precisión relativa r = {r_n:.1%}')
print(f'\nRef. estacionaria Wq = {Wq_ref:.3f} (¿dentro del IC? {"Sí" if ci[0] <= Wq_ref <= ci[1] else "No"})')

In [ ]:
# Demostración de autocorrelación DENTRO de una corrida
np.random.seed(42)
esperas_una_corrida = []
def cliente_det(env, srv):
    t_arr = env.now
    with srv.request() as req:
        yield req
        esperas_una_corrida.append(env.now - t_arr)
        yield env.timeout(np.random.exponential(1/mu))
def llegadas_det(env, srv):
    while True:
        yield env.timeout(np.random.exponential(1/lam))
        env.process(cliente_det(env, srv))
env = simpy.Environment()
srv = simpy.Resource(env, capacity=1)
env.process(llegadas_det(env, srv))
env.run(until=T_turno)
W = np.array(esperas_una_corrida)

# Autocorrelación
from numpy import correlate
def autocorr(x, maxlag=30):
    x_c = x - x.mean()
    r = np.correlate(x_c, x_c, 'full')
    r = r[len(r)//2:] / r[len(r)//2]
    return r[:maxlag+1]

rho_k = autocorr(W, 30)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Serie de tiempo de esperas
axes[0].plot(W, color='#1A7A9A', lw=0.8, alpha=0.7)
axes[0].axhline(W.mean(), color='#C62828', lw=2, ls='--', label=f'Media = {W.mean():.2f}')
axes[0].set_xlabel('Cliente i'); axes[0].set_ylabel('Tiempo de espera (min)')
axes[0].set_title('Esperas dentro de UNA corrida'); axes[0].legend()

# Autocorrelograma
axes[1].bar(range(len(rho_k)), rho_k, color='#1A7A9A', alpha=0.8)
axes[1].axhline(2/np.sqrt(len(W)), ls='--', color='#C62828', label='±2/√n')
axes[1].axhline(-2/np.sqrt(len(W)), ls='--', color='#C62828')
axes[1].set_xlabel('Lag k'); axes[1].set_ylabel('ρ(k)')
axes[1].set_title('Autocorrelación → NO independientes'); axes[1].legend(fontsize=8)

# Réplicas (independientes)
axes[2].bar(range(1, n+1), Y, color='#1A7A9A', alpha=0.8)
axes[2].axhline(Ybar, color='#C62828', lw=2, ls='--', label=f'Ȳ = {Ybar:.3f}')
axes[2].axhspan(ci[0], ci[1], alpha=0.15, color='#C62828', label='IC 95%')
axes[2].set_xlabel('Réplica j'); axes[2].set_ylabel('Yj (min)')
axes[2].set_title('Réplicas → SÍ independientes'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f'ρ(1) dentro de la corrida: {rho_k[1]:.3f} — autocorrelación significativa')
print(f'Usar S/√m subestimaría el error estándar real')

## Ejemplo 2 — Banco M/E₂/1 y cálculo de n*
<div class='defn'>
Mismo banco pero con servicio Erlang-2 (CV=0.71). Se quiere estimar Wq con precisión δ=0.05 min al 95%.
</div>

**Procedimiento secuencial:**
1. Piloto de n₀ ≥ 10 réplicas → obtener S²
2. Calcular n* = ⌈t²·S² / δ²⌉
3. Si n* ≤ n₀, ya terminamos
4. Si n* > n₀, ejecutar réplicas adicionales

In [ ]:
def simular_turno_me2(lam, mu_fase, T, seed):
    """Simula turno terminal M/E₂/1."""
    np.random.seed(seed)
    esperas = []
    
    def cliente(env, srv):
        t_arr = env.now
        with srv.request() as req:
            yield req
            esperas.append(env.now - t_arr)
            yield env.timeout(np.random.exponential(1/mu_fase))
            yield env.timeout(np.random.exponential(1/mu_fase))
    
    def llegadas(env, srv):
        while True:
            yield env.timeout(np.random.exponential(1/lam))
            env.process(cliente(env, srv))
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)
    env.process(llegadas(env, srv))
    env.run(until=T)
    return np.mean(esperas) if esperas else 0

mu_fase_e2 = 2 * mu  # mu=3, mu_fase=6 para E[S]=2/6=1/3 min
delta = 0.05  # precisión absoluta deseada

# Paso 1: Piloto de n₀=15 réplicas
n0 = 15
Y_pilot = np.array([simular_turno_me2(lam, mu_fase_e2, T_turno, seed=500+j) for j in range(n0)])
Ybar_p = Y_pilot.mean()
S_p = Y_pilot.std(ddof=1)
t_p = st.t.ppf(0.975, n0-1)
h_p = t_p * S_p / np.sqrt(n0)

print('═══ EJEMPLO 2: M/E₂/1 — PROCEDIMIENTO SECUENCIAL ═══')
print(f'\nPiloto (n₀={n0}): Ȳ = {Ybar_p:.3f} min, S = {S_p:.4f} min')
print(f'Semiancho actual: h = {h_p:.4f} min (objetivo δ = {delta:.2f})')

# Paso 2: Calcular n*
n_star = int(np.ceil((t_p**2 * S_p**2) / delta**2))
n_star = max(n_star, n0)
print(f'\nn* = ⌈({t_p:.3f}² × {S_p:.4f}²) / {delta}²⌉ = {int(np.ceil((t_p**2 * S_p**2) / delta**2))}')

if n_star <= n0:
    print(f'n* = {n_star} ≤ n₀ = {n0} → ¡el piloto es suficiente!')
    print(f'h = {h_p:.4f} < δ = {delta:.4f} ✓')
else:
    print(f'n* = {n_star} > n₀ = {n0} → ejecutar {n_star - n0} réplicas adicionales')
    Y_extra = np.array([simular_turno_me2(lam, mu_fase_e2, T_turno, seed=600+j) for j in range(n_star - n0)])
    Y_all = np.concatenate([Y_pilot, Y_extra])
    Ybar_all = Y_all.mean()
    S_all = Y_all.std(ddof=1)
    h_all = st.t.ppf(0.975, n_star-1) * S_all / np.sqrt(n_star)
    print(f'Con n={n_star}: Ȳ = {Ybar_all:.3f}, h = {h_all:.4f} {"✓" if h_all <= delta else "→ repetir"}')

# Referencia P-K
ES_e2 = 1/mu  # 1/3 min
ES2_e2 = 2/mu_fase_e2**2 + ES_e2**2
rho_e2 = lam * ES_e2
Wq_PK_e2 = lam * ES2_e2 / (2*(1-rho_e2))
print(f'\nRef. P-K (estacionaria): Wq = {Wq_PK_e2:.3f} min')

# Comparación con M/M/1
print(f'\n═══ COMPARACIÓN ═══')
print(f'{"Modelo":<12} {"Ȳ (min)":>10} {"S (min)":>10} {"IC 95%":>20}')
print('-'*55)
print(f'{"M/M/1":<12} {Ybar:>10.3f} {S_n:>10.3f} [{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'{"M/E₂/1":<12} {Ybar_p:>10.3f} {S_p:>10.3f} [{Ybar_p-h_p:.3f}, {Ybar_p+h_p:.3f}]')
print(f'\nReducción de Wq: {(1-Ybar_p/Ybar)*100:.0f}% (Erlang reduce variabilidad → menor espera)')

## Ejemplo 3 — Urgencias hospitalarias
<div class='defn'>
Turno de mañana 7:00–15:00 (8h). 3 médicos, cola con prioridad no expulsiva.

| Nivel | Fracción | Servicio | E[S] | CV |
|---|---|---|---|---|
| P1 (urgente) | 15% | Exp(μ=0.1) | 10 min | 1.00 |
| P2 (moderado) | 55% | Erlang-3(μf=0.1) | 30 min | 0.58 |
| P3 (leve) | 30% | LogNormal(μL=3.4, σL=0.5) | 35 min | 0.53 |

λ = 20 pac/h. **No existe solución analítica** → solo simulación.
</div>

Métricas: W_q(P2), P(W_q>30 min | P3), ρ — con precisión r ≤ 5%.

In [ ]:
def urgencias(lam, T, c, seed):
    """Simula urgencias con prioridades no expulsivas."""
    np.random.seed(seed)
    esperas = {1: [], 2: [], 3: []}
    tiempos_ocup = []
    
    def gen_servicio(prioridad):
        if prioridad == 1:
            return np.random.exponential(10)
        elif prioridad == 2:
            return sum(np.random.exponential(10) for _ in range(3))  # Erlang-3
        else:
            return np.random.lognormal(3.4, 0.5)
    
    def paciente(env, srv, prio):
        t_arr = env.now
        with srv.request(priority=prio) as req:
            yield req
            esperas[prio].append(env.now - t_arr)
            t_srv = gen_servicio(prio)
            tiempos_ocup.append(t_srv)
            yield env.timeout(t_srv)
    
    def llegadas(env, srv):
        while True:
            yield env.timeout(np.random.exponential(60/lam))  # min
            u = np.random.random()
            prio = 1 if u < 0.15 else (2 if u < 0.70 else 3)
            env.process(paciente(env, srv, prio))
    
    env = simpy.Environment()
    srv = simpy.PriorityResource(env, capacity=c)
    env.process(llegadas(env, srv))
    env.run(until=T)
    
    Wq_P2 = np.mean(esperas[2]) if esperas[2] else 0
    prob_P3 = np.mean([w > 30 for w in esperas[3]]) if esperas[3] else 0
    rho = sum(tiempos_ocup) / (T * c)
    return Wq_P2, prob_P3, rho

T_urg = 480  # 8 horas en minutos
lam_urg, c_urg = 20, 3

# ─── Piloto: 10 réplicas ───
n0_u = 10
results_pilot = np.array([urgencias(lam_urg, T_urg, c_urg, seed=800+j) for j in range(n0_u)])
Wq_P2_pilot = results_pilot[:, 0]
prob_P3_pilot = results_pilot[:, 1]
rho_pilot = results_pilot[:, 2]

print('═══ PILOTO (n₀=10) ═══')
metrics = [('Wq(P2) min', Wq_P2_pilot), ('P(Wq>30|P3)', prob_P3_pilot), ('ρ', rho_pilot)]
for name, data in metrics:
    m, s = data.mean(), data.std(ddof=1)
    r = st.t.ppf(0.975, n0_u-1) * s / (np.sqrt(n0_u) * abs(m)) if m != 0 else float('inf')
    print(f'  {name:<14}: Ȳ = {m:.3f}, S = {s:.3f}, r = {r:.1%}')

# ─── Calcular n* para cada métrica (r ≤ 5%) ───
print(f'\n═══ CÁLCULO DE n* (objetivo r ≤ 5%) ═══')
t_p10 = st.t.ppf(0.975, n0_u-1)
n_stars = []
for name, data in metrics:
    m, s = data.mean(), data.std(ddof=1)
    n_s = int(np.ceil((t_p10**2 * s**2) / (0.05 * m)**2)) if m != 0 else 100
    n_stars.append(n_s)
    print(f'  {name:<14}: n* = {n_s}')
n_final = max(max(n_stars), n0_u)
print(f'  → n* final = {n_final} (la métrica más exigente)')

In [ ]:
# ─── Ejecutar n* réplicas completas ───
n_final = max(n_final, 40)  # mínimo 40 para robustez
results_full = np.array([urgencias(lam_urg, T_urg, c_urg, seed=900+j) for j in range(n_final)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
titles = ['Wq(P2) — Espera pacientes moderados', 'P(Wq>30 min | P3)', 'ρ — Utilización médicos']
colors_u = ['#1A7A9A', '#C8961E', '#C62828']

print(f'═══ RESULTADOS FINALES (n={n_final} réplicas) ═══')
print(f'{"Métrica":<18} {"Ȳ":>8} {"IC 95%":>20} {"r":>8}')
print('-'*58)

for i, (name, title, col) in enumerate(zip(['Wq(P2) min', 'P(Wq>30|P3)', 'ρ'], titles, colors_u)):
    data = results_full[:, i]
    m = data.mean()
    s = data.std(ddof=1)
    t_c = st.t.ppf(0.975, n_final-1)
    h = t_c * s / np.sqrt(n_final)
    r = h / abs(m) if m != 0 else float('inf')
    ci_lo, ci_hi = m - h, m + h
    
    print(f'{name:<18} {m:>8.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {r:>8.1%}')
    
    axes[i].hist(data, bins=12, color=col, edgecolor='white', alpha=0.8)
    axes[i].axvline(m, color='black', lw=2, label=f'Ȳ = {m:.3f}')
    axes[i].axvspan(ci_lo, ci_hi, alpha=0.15, color=col, label=f'IC 95%')
    axes[i].set_xlabel(name); axes[i].set_title(title, fontsize=9)
    axes[i].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'\n📊 Interpretación:')
print(f'  • Pacientes P2 esperan ~{results_full[:,0].mean():.0f} min en promedio')
print(f'  • {results_full[:,1].mean()*100:.0f}% de P3 esperan >30 min')
print(f'  • Utilización médicos: {results_full[:,2].mean()*100:.0f}% (cercano a saturación)')

## Conclusiones

- Las observaciones **dentro** de una corrida están autocorrelacionadas → no usar S/√m.
- El método de **réplicas independientes** genera Y₁,...,Yₙ i.i.d. → estadística clásica (t-Student).
- El **procedimiento secuencial** determina cuántas réplicas se necesitan para una precisión deseada.
- Distribuciones con **menor CV** (Erlang vs. Exponencial) producen estimadores más precisos con menos réplicas.
- Para sistemas complejos (urgencias con prioridades, servicios no exponenciales), la **simulación es la única herramienta** — no existe solución analítica.
- Siempre reportar Ȳ **con su IC**, no solo la media.

**Próximo tema:** T3.4 — Simulaciones de estado estacionario: warm-up, Batch Means y CRN.